<a href="https://colab.research.google.com/github/keyvanmahmoudi/Deep_Learning_Course/blob/main/solutions/Session11_lstm_and_text_classification_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۱۱: LSTM، GRU و Word Embedding برای دسته‌بندی متن

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session11_lstm_and_text_classification.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

> **نکتهٔ اجرا:** بعضی پاسخ‌ها مدل‌های بزرگ‌تری آموزش می‌دهند؛ در Colab از منوی `Runtime → Change runtime type` گزینهٔ **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
%pip install -q datasets gensim

In [ ]:
import re
import time
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
PAD, UNK = 0, 1

imdb = load_dataset('stanfordnlp/imdb')
train_texts, train_labels = imdb['train']['text'], imdb['train']['label']
test_texts, test_labels = imdb['test']['text'], imdb['test']['label']
TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?")


def tokenize(text):
    return TOKEN_RE.findall(text.lower().replace('<br />', ' '))


def build_vocab(texts, tokenizer, max_vocab=20_000):
    counter = Counter(tok for t in texts for tok in tokenizer(t))
    itos = ['<pad>', '<unk>'] + [w for w, _ in counter.most_common(max_vocab - 2)]
    return itos, {w: i for i, w in enumerate(itos)}


class ReviewDataset(Dataset):
    def __init__(self, texts, labels, stoi, tokenizer, max_len):
        self.data = [torch.tensor([stoi.get(t, UNK) for t in tokenizer(x)][:max_len] or [UNK]) for x in texts]
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        return self.data[i], self.labels[i]


def collate(batch):
    seqs, labels = zip(*batch)
    return pad_sequence(seqs, batch_first=True, padding_value=PAD), torch.tensor([len(s) for s in seqs]), torch.stack(labels)


def make_loaders(max_len, texts=(train_texts, test_texts), labels=(train_labels, test_labels), tokenizer=tokenize):
    itos, stoi = build_vocab(texts[0], tokenizer)
    tr = ReviewDataset(texts[0], labels[0], stoi, tokenizer, max_len)
    te = ReviewDataset(texts[1], labels[1], stoi, tokenizer, max_len)
    return (DataLoader(tr, batch_size=64, shuffle=True, collate_fn=collate),
            DataLoader(te, batch_size=256, collate_fn=collate), itos, stoi)


class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden=128, n_classes=2, pooling='last', embedding=None):
        super().__init__()
        self.pooling = pooling
        self.embedding = embedding if embedding is not None else nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.rnn = nn.LSTM(self.embedding.embedding_dim, hidden, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(2 * hidden, n_classes)

    def forward(self, x, lengths):
        e = self.dropout(self.embedding(x))
        packed = pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, (h_n, _) = self.rnn(packed)
        if self.pooling == 'last':
            feat = torch.cat([h_n[-2], h_n[-1]], dim=1)
        else:                                                     # max pooling over time
            out, _ = pad_packed_sequence(out, batch_first=True, padding_value=float('-inf'))
            feat = out.max(dim=1).values                          # padded positions are -inf, never the max
        return self.fc(self.dropout(feat))


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    correct = n = 0
    with torch.set_grad_enabled(training):
        for X, L, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X, L)
            if training:
                loss = nn.functional.cross_entropy(logits, y)
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
            correct += (logits.argmax(1) == y).sum().item(); n += len(y)
    return correct / n


def train_eval(model, train_loader, test_loader, epochs=5, lr=2e-3, label=''):
    model = model.to(device)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    start, accs = time.time(), []
    for _ in range(epochs):
        run_epoch(model, train_loader, opt)
        accs.append(run_epoch(model, test_loader))
    print(f'{label:32} test acc per epoch {np.round(accs, 4)} | {time.time() - start:.0f}s')
    return model, accs


train_loader, test_loader, itos, stoi = make_loaders(256)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

`MyGRUCell` را از صفر پیاده‌سازی و با `nn.GRUCell` تطبیق دهید (ترتیب دروازه‌ها در PyTorch: reset، update، new).

</div>

In [ ]:
class MyGRUCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.x2g = nn.Linear(input_size, 3 * hidden_size)        # reset, update, new
        self.h2g = nn.Linear(hidden_size, 3 * hidden_size)

    def forward(self, x, h):
        x_r, x_z, x_n = self.x2g(x).chunk(3, dim=1)
        h_r, h_z, h_n = self.h2g(h).chunk(3, dim=1)
        r = torch.sigmoid(x_r + h_r)                              # reset gate
        z = torch.sigmoid(x_z + h_z)                              # update gate
        n = torch.tanh(x_n + r * h_n)                             # candidate: reset applied to the *hidden* part
        return (1 - z) * n + z * h                                # interpolation between new and old state


ref, mine = nn.GRUCell(3, 4), MyGRUCell(3, 4)
with torch.no_grad():
    mine.x2g.weight.copy_(ref.weight_ih); mine.x2g.bias.copy_(ref.bias_ih)
    mine.h2g.weight.copy_(ref.weight_hh); mine.h2g.bias.copy_(ref.bias_hh)
x, h = torch.randn(5, 3), torch.randn(5, 4)
print('outputs match:', torch.allclose(ref(x, h), mine(x, h), atol=1e-6))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

دو جزئیات پیاده‌سازی PyTorch که باید رعایت شوند:

- دروازهٔ Reset در **داخل** جمله، در $W_{hn}\mathbf{h} + \mathbf{b}_{hn}$ ضرب می‌شود: $\mathbf{n} = \tanh(W_{in}\mathbf{x} + \mathbf{b}_{in} + \mathbf{r} \odot (W_{hn}\mathbf{h} + \mathbf{b}_{hn}))$. به همین دلیل بایاس‌های ورودی و پنهان را جدا نگه داشتیم.
- در PyTorch خروجی $(1 - \mathbf{z})\odot\mathbf{n} + \mathbf{z}\odot\mathbf{h}$ است، یعنی $\mathbf{z}$ نزدیک ۱ یعنی «حالت قبلی را نگه دار». بعضی منابع نقش $\mathbf{z}$ و $1-\mathbf{z}$ را برعکس می‌نویسند؛ از نظر ریاضی معادل‌اند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

`MAX_LEN` را ۶۴ و ۵۱۲ بگیرید. دقت و زمان آموزش چه تغییری می‌کند؟

</div>

In [ ]:
for max_len in (64, 256, 512):
    tr, te, it, _ = make_loaders(max_len)
    torch.manual_seed(0)
    train_eval(RNNClassifier(len(it)), tr, te, label=f'MAX_LEN = {max_len}')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

میانهٔ طول نقدها حدود ۱۷۰ Token است. با `MAX_LEN = 64` بخش بزرگی از متن حذف می‌شود، از جمله جمع‌بندی پایانی که اغلب صریح‌ترین قضاوت را دارد. پس دقت چند درصد پایین می‌آید، ولی آموزش بسیار سریع‌تر است. با ۵۱۲ تقریباً همهٔ متن دیده می‌شود، اما زمان آموزش تقریباً متناسب با طول بیشتر می‌شود. سود دقت نسبت به ۲۵۶ هم کم است، چون فقط حدود ۱۰٪ نقدها بلندتر از ۴۰۰ Token‌اند و LSTM هم وابستگی‌های خیلی دور را به‌سختی حفظ می‌کند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

به‌جای حالت نهایی، **Max Pooling روی زمان** (بیشینه خروجی‌های LSTM در همه گام‌ها) را به طبقه‌بند بدهید. برای این کار از `pad_packed_sequence` استفاده کنید و مراقب مکان‌های Padding باشید.

</div>

In [ ]:
for pooling in ('last', 'max'):
    torch.manual_seed(0)
    train_eval(RNNClassifier(len(itos), pooling=pooling), train_loader, test_loader, label=f'pooling = {pooling}')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

`pad_packed_sequence` خروجی فشرده را به یک تنسور `(batch, max_len, 2*hidden)` برمی‌گرداند و جای Tokenهای Padding را با `padding_value` پر می‌کند. اگر مقدار پیش‌فرض صفر را بگذاریم و خروجی‌های واقعی همه منفی باشند، بیشینه اشتباهاً صفرِ Padding می‌شود؛ پس از $-\infty$ استفاده کردیم.

Max Pooling به‌جای اتکا به یک حالت نهایی، **قوی‌ترین سیگنال هر ویژگی در کل نقد** را برمی‌دارد. اگر جملهٔ کلیدی در وسط یک نقد طولانی باشد، لازم نیست LSTM آن را تا انتها در حافظه نگه دارد. معمولاً این روش کمی بهتر و سریع‌تر همگرا می‌شود.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

Embeddingهای ازپیش‌آموزش‌دیده GloVe (۱۰۰ بعدی) را دانلود و در لایه Embedding بارگذاری کنید (`nn.Embedding.from_pretrained`). یک بار منجمد و یک بار قابل آموزش امتحان کنید.

</div>

In [ ]:
import gensim.downloader as api

glove = api.load('glove-wiki-gigaword-100')                   # 400k words, 100-d (about 130 MB)
weights = torch.randn(len(itos), 100) * 0.1
found = 0
for i, word in enumerate(itos):
    if word in glove.key_to_index:
        weights[i] = torch.tensor(glove[word]); found += 1
weights[PAD] = 0
print(f'{found:,} of {len(itos):,} vocabulary words found in GloVe ({found / len(itos):.1%})')

for freeze in (True, False):
    torch.manual_seed(0)
    emb = nn.Embedding.from_pretrained(weights.clone(), freeze=freeze, padding_idx=PAD)
    train_eval(RNNClassifier(len(itos), embedding=emb), train_loader, test_loader,
               label=f'GloVe, {"frozen" if freeze else "trainable"}')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

کتابخانهٔ `gensim` بردارهای GloVe را آماده دانلود می‌کند. کلماتی که در GloVe نیستند (بیشتر غلط‌های املایی و نام‌های خاص) با بردار تصادفی کوچک شروع می‌کنند.

- **منجمد:** Embeddingها از دانش عمومی زبان (شباهت معنایی روی ویکی‌پدیا) می‌آیند و همگرایی در Epochهای اول سریع‌تر است. اما برای احساس بهینه نیستند؛ مثلاً *good* و *bad* در GloVe نزدیک‌اند، چون در بافت‌های مشابه به کار می‌روند.
- **قابل آموزش:** بهترین هر دو حالت است: از نقطهٔ شروع خوب GloVe آغاز می‌شود و برای احساس تنظیم می‌شود. با ۲۵ هزار نقد، تفاوت با Embedding تصادفی کوچک است. مزیت Embeddingهای ازپیش‌آموزش‌دیده با دادهٔ کم بیشتر دیده می‌شود.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۵

همین خط لوله را روی یک دیتاست احساس فارسی (مثلاً نظرات کاربران دیجی‌کالا یا اسنپ‌فود از Hugging Face Hub) اجرا کنید. چه تغییری در Tokenizer لازم است؟

</div>

In [ ]:
dksf = load_dataset('hezarai/sentiment-dksf')                 # labels: 0 negative, 1 positive, 2 neutral
FA_TOKEN_RE = re.compile(r'[؀-ۿ‌]+|[a-zA-Z]+|\d+')
ARABIC_TO_PERSIAN = str.maketrans({'ي': 'ی', 'ك': 'ک', 'ة': 'ه', 'ۀ': 'ه'})
DIACRITICS = re.compile(r'[ً-ْ]')


def tokenize_fa(text):
    text = DIACRITICS.sub('', text.translate(ARABIC_TO_PERSIAN))      # unify letters, drop short vowels
    text = text.replace('‌', ' ')                              # simplest choice: split on ZWNJ
    return FA_TOKEN_RE.findall(text.lower())


print(tokenize_fa('این گوشي واقعاً عالی‌ست ولی باتری‌اش ضعیفه!'))
fa_train, fa_test, fa_itos, _ = make_loaders(
    128, texts=(dksf['train']['text'], dksf['test']['text']),
    labels=(dksf['train']['label'], dksf['test']['label']), tokenizer=tokenize_fa)
torch.manual_seed(0)
fa_model, _ = train_eval(RNNClassifier(len(fa_itos), n_classes=3), fa_train, fa_test, label='Persian reviews (3 classes)')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

تغییرات لازم در Tokenizer برای فارسی:

- **بازهٔ یونیکد:** الگوی `[a-z0-9]` فقط حروف لاتین را می‌گیرد؛ حروف فارسی در بازهٔ `؀-ۿ` هستند.
- **یکسان‌سازی نویسه‌ها:** «ي» و «ك» عربی باید به «ی» و «ک» فارسی تبدیل شوند؛ وگرنه یک کلمه دو Token متفاوت می‌شود. اعراب هم حذف می‌شوند.
- **نیم‌فاصله (ZWNJ):** «باتری‌اش» یا «می‌خواهم» با نیم‌فاصله نوشته می‌شوند، اما کاربران اغلب فاصله یا هیچ‌چیز می‌گذارند. ساده‌ترین راه شکستن روی نیم‌فاصله است. برای کار جدی از نرمال‌ساز کتابخانه‌های **hazm** یا **parsivar** استفاده کنید.
- **حروف بزرگ و کوچک** در فارسی معنا ندارد و `lower()` فقط روی بخش لاتین اثر می‌گذارد.

دیتاست سه برچسب دارد، پس `n_classes=3`. کلاس خنثی سخت‌ترین است؛ در جلسهٔ ۱۴ همین داده را با ParsBERT حل می‌کنیم و با این خط مبنا مقایسه می‌کنیم.

</div>